In [99]:
import pandas as pd
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import r2_score, mean_squared_error


from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from scipy.stats import spearmanr

X_df =pd.read_csv('C:/Users/Julien GILLES/Documents/ESILV A4 Machine learning Projet/X_train_NHkHMNU.csv', delimiter= ',')
y_df =pd.read_csv('C:/Users/Julien GILLES/Documents/ESILV A4 Machine learning Projet/y_train_ZAN5mwg.csv', delimiter= ',')
X_test_df =pd.read_csv('C:/Users/Julien GILLES/Documents/ESILV A4 Machine learning Projet/X_test_final.csv', delimiter= ',')
df = pd.merge(X_df,y_df,on='ID')

## I. FONCTION EN UNE FOIS

In [115]:
def Model_Final(df,model_fr, model_de):
    #1. 
    #Split the original dataset in 2 datasets : one for the rows with COUNTRY = FR and one for the rows with COUNTRY = DE
    # For FR :
    df_fr = df[df['COUNTRY'] == 'FR']
    # For DE :
    df_de = df[df['COUNTRY'] == 'DE']

    #2.
    #Elimination des colonnes qui ne concernent pas directement le pays
    columns_to_drop_fr = ['COUNTRY','DE_CONSUMPTION', 'DE_FR_EXCHANGE', 'FR_NET_EXPORT','DE_NET_EXPORT','DE_NET_IMPORT','DE_GAS','DE_COAL','DE_HYDRO','DE_NUCLEAR','DE_SOLAR','DE_WINDPOW','DE_LIGNITE','DE_RESIDUAL_LOAD','DE_RAIN','DE_WIND','DE_TEMP']
    df_fr = df_fr.drop(columns=columns_to_drop_fr)
    columns_to_drop_de = ['COUNTRY','FR_CONSUMPTION', 'FR_DE_EXCHANGE','DE_NET_EXPORT', 'FR_NET_EXPORT','FR_NET_IMPORT','FR_GAS','FR_COAL','FR_HYDRO','FR_NUCLEAR','FR_SOLAR','FR_WINDPOW', 'FR_RESIDUAL_LOAD','FR_RAIN','FR_WIND','FR_TEMP']
    df_de = df_de.drop(columns=columns_to_drop_de)

    #3.
    #Remplacement des valeurs manquantes par la moyenne de leurs colonnes dans chaque dataset 
    df_fr, df_de = [df.fillna(df.mean()) for df in [df_fr, df_de]]

    #5.
    #Implémentation du modèle pour FR
    X_fr = df_fr.drop(columns=['ID', 'TARGET'])
    y_fr = df_fr['TARGET']
    
    X_train_fr, X_test_fr, y_train_fr, y_test_fr = train_test_split(X_fr, y_fr, test_size=0.2, random_state=42)
    #standardisation :
    scaler = StandardScaler()
    X_train_scaled_fr = scaler.fit_transform(X_train_fr)
    X_test_scaled_fr = scaler.transform(X_test_fr)

    model_fr.fit(X_train_scaled_fr, y_train_fr)

    y_pred_fr= model_fr.predict(X_test_scaled_fr)

    #6.
    #Implémentation du modèles pour DE
    X_de = df_de.drop(columns=['ID', 'TARGET'])
    y_de = df_de['TARGET']
    
    X_train_de, X_test_de, y_train_de, y_test_de = train_test_split(X_de, y_de, test_size=0.2, random_state=42)
    #standardisation :
    scaler = StandardScaler()
    X_train_scaled_de = scaler.fit_transform(X_train_de)
    X_test_scaled_de = scaler.transform(X_test_de)

    model_de.fit(X_train_scaled_de, y_train_de)

    y_pred_de = model_de.predict(X_test_scaled_de)

    #7.
    #Analyse des résultats
    #pour FR :
    
    #score R2 - FR
    score_fr = model.score(X_test_scaled_fr, y_test_fr)
    print(f"R2 score FR: {score_fr}")

    #score Spearman - FR
    spearman = spearmanr(y_test_fr, y_pred_fr)
    print(f"Spearman score FR: {spearman.correlation}")

    #MSE - FR
    mse_fr = mean_squared_error(y_test_fr, y_pred_fr)
    print(f"MSE FR: {mse_fr}")
    
    #pour DE :
    
    #score R2 - DE
    score_de = model_de.score(X_test_scaled_de, y_test_de)
    print(f"R2 score DE: {score_de}")

    #score Spearman - DE
    spearman = spearmanr(y_test_de, y_pred_de)
    print(f"Spearman score DE: {spearman.correlation}")

    #MSE - DE
    mse_de = mean_squared_error(y_test_de, y_pred_de)
    print(f"MSE DE: {mse_de}")

    #8.
    # Fusionner les résultats
    # Concaténer les prédictions et les vrais résultats pour FR et DE
    y_true_combined = pd.concat([y_test_fr, y_test_de], axis=0)
    y_pred_combined = pd.concat([pd.Series(y_pred_fr), pd.Series(y_pred_de)], axis=0)

    # Calculer les scores pour la combinaison
    combined_r2 = r2_score(y_true_combined, y_pred_combined)
    print(f"Combined R2 score: {combined_r2}")

    combined_spearman = spearmanr(y_true_combined, y_pred_combined)
    print(f"Combined Spearman score: {combined_spearman.correlation}")

    combined_mse = mean_squared_error(y_true_combined, y_pred_combined)
    print(f"Combined MSE: {combined_mse}")



In [116]:
Model_Final(df,LinearRegression(),LinearRegression())

R2 score FR: 0.001308843996675746
Spearman score FR: 0.12196349747014068
MSE FR: 1.4081039065999932
R2 score DE: 0.05029220088485209
Spearman score DE: 0.4148423524150269
MSE DE: 0.9050128222656999
Combined R2 score: 0.019226057856309597
Combined Spearman score: 0.2760418564837022
Combined MSE: 1.191774740336247


## II. DIVISION FONCTION

## 1. split_dataset_by_country

##### As seen during our analysis of the original dataset, we observed that it would be more accurate to develop a model separately for France and Germany. The function split_dataset_by_country, which takes the original dataset as a parameter, enables splitting the dataset into two new datasets: df_fr (for the rows concerning France) and df_de (for the rows concerning Germany).

In [117]:
def split_dataset_by_country(df):
    df_fr = df[df['COUNTRY'] == 'FR']
    df_de = df[df['COUNTRY'] == 'DE']
    return df_fr, df_de

## 2. drop_irrelevant_columns

##### We decided, for each country (and thus for each model), to focus on the columns specifically related to each country. The function drop_irrelevant_columns allows us to remove the columns related to Germany from the df_fr dataset and the columns related to France from the df_de dataset.

In [118]:
def drop_irrelevant_columns(df, country):
    if country == 'FR':
        columns_to_drop = ['COUNTRY', 'DE_CONSUMPTION', 'DE_FR_EXCHANGE', 'FR_NET_EXPORT',
                           'DE_NET_EXPORT', 'DE_NET_IMPORT', 'DE_GAS', 'DE_COAL', 'DE_HYDRO',
                           'DE_NUCLEAR', 'DE_SOLAR', 'DE_WINDPOW', 'DE_LIGNITE', 'DE_RESIDUAL_LOAD',
                           'DE_RAIN', 'DE_WIND', 'DE_TEMP']
    elif country == 'DE':
        columns_to_drop = ['COUNTRY', 'FR_CONSUMPTION', 'FR_DE_EXCHANGE', 'DE_NET_EXPORT',
                           'FR_NET_EXPORT', 'FR_NET_IMPORT', 'FR_GAS', 'FR_COAL', 'FR_HYDRO',
                           'FR_NUCLEAR', 'FR_SOLAR', 'FR_WINDPOW', 'FR_RESIDUAL_LOAD',
                           'FR_RAIN', 'FR_WIND', 'FR_TEMP']
    return df.drop(columns=columns_to_drop)

## 3. fill_missing_values

##### Each dataset contains a certain number of missing values (as previously noted), and given the relatively limited amount of data, we couldn’t simply remove the rows with missing values. Therefore, we use the fill_missing_values function on df_fr and df_de individually to replace the missing values with the average of the column where the missing value is located.

In [119]:
def fill_missing_values(df):
    return df.fillna(df.mean())

## 4. prepare_data

##### The prepare_data function prepares the data for training and testing the model. It separates the features (X) from the target (y), splits the data into training and testing sets, and standardizes the features by applying scaling with StandardScaler.

In [120]:
def prepare_data(df):
    X = df.drop(columns=['ID', 'TARGET'])
    y = df['TARGET']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled, y_train, y_test

## 5. train_and_predict

##### Now that our data is ready, we can train our model on it and make our predictions. The train_and_predict function trains a given model using the training data (X_train and y_train), then makes predictions on the test data (X_test). It returns the predicted values (y_pred).

In [121]:
def train_and_predict(model, X_train, y_train, X_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return y_pred

## 6. evaluate_model

##### Our predictions are made: it is now possible to evaluate our model. The evaluate_model function provides several evaluation metrics, including the R2 score, MSE, and Spearman's correlation.

def evaluate_model(y_test, y_pred, model, X_test):
    r2 = model.score(X_test, y_test)
    mse = mean_squared_error(y_test, y_pred)
    spearman_corr = spearmanr(y_test, y_pred).correlation
    return r2, mse, spearman_corr

## 7. combine_results

##### Although we trained models and made predictions separately for France and Germany, the final model to submit at the end of the challenge is a single model. The function combine_results allows us to test the previous metrics, but this time on the entire set of our predictions.

In [122]:
def combine_results(y_test_fr, y_pred_fr, y_test_de, y_pred_de):
    y_true_combined = pd.concat([y_test_fr, y_test_de], axis=0)
    y_pred_combined = pd.concat([pd.Series(y_pred_fr), pd.Series(y_pred_de)], axis=0)
    combined_r2 = r2_score(y_true_combined, y_pred_combined)
    combined_mse = mean_squared_error(y_true_combined, y_pred_combined)
    combined_spearman = spearmanr(y_true_combined, y_pred_combined).correlation
    return combined_r2, combined_mse, combined_spearman

## 8. Model_Final

##### Finally, we implement our global function that uses the previously defined functions 1 to 7.

In [123]:
def Model_Final(df, model_fr, model_de):
    # Step 1 : datasets splitting
    df_fr, df_de = split_dataset_by_country(df)

    #Step 2: Dropping unnecessary columns"    
    df_fr = drop_irrelevant_columns(df_fr, 'FR')
    df_de = drop_irrelevant_columns(df_de, 'DE')

    # Step 3 : Missing values replacement
    df_fr = fill_missing_values(df_fr)
    df_de = fill_missing_values(df_de)

    # Step 4 : Data preparation for FR and DE
    X_train_fr, X_test_fr, y_train_fr, y_test_fr = prepare_data(df_fr)
    X_train_de, X_test_de, y_train_de, y_test_de = prepare_data(df_de)

    # Step 5 : Training et predicting for FR
    y_pred_fr = train_and_predict(model_fr, X_train_fr, y_train_fr, X_test_fr)

    # Step 6 : Training and predicting for DE
    y_pred_de = train_and_predict(model_de, X_train_de, y_train_de, X_test_de)

    # Step 7 : Models evaluation
    score_fr, mse_fr, spearman_fr = evaluate_model(y_test_fr, y_pred_fr, model_fr, X_test_fr)
    print(f"FR - R2: {score_fr}, MSE: {mse_fr}, Spearman: {spearman_fr}")

    score_de, mse_de, spearman_de = evaluate_model(y_test_de, y_pred_de, model_de, X_test_de)
    print(f"DE - R2: {score_de}, MSE: {mse_de}, Spearman: {spearman_de}")

    # Step 8 : Combining the results and evaluating
    combined_r2, combined_mse, combined_spearman = combine_results(y_test_fr, y_pred_fr, y_test_de, y_pred_de)
    print(f"Combined - R2: {combined_r2}, MSE: {combined_mse}, Spearman: {combined_spearman}")

In [124]:
Model_Final(df,LinearRegression(),LinearRegression())

FR - R2: 0.001308843996675746, MSE: 1.4081039065999932, Spearman: 0.12196349747014068
DE - R2: 0.05029220088485209, MSE: 0.9050128222656999, Spearman: 0.4148423524150269
Combined - R2: 0.019226057856309597, MSE: 1.191774740336247, Spearman: 0.2760418564837022
